# Parte 1 — Análise exploratória com APIs externas

Clima (Open-Meteo), feriados (Nager.Date), padrões geoespaciais e demanda do 1746 (2023–2024).

## Instalando bibliotecas necessárias

## Importando bibliotecas necessárias

In [1]:
import basedosdados as bd
import pandas as pd
import requests

### Configurações

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Base de dados de chamados

### Baixando dados da base

In [3]:
df_chamados = bd.read_sql(
    "SELECT * FROM `datario.adm_central_atendimento_1746.chamado` WHERE data_particao >= '2023-01-01' AND data_particao <= '2024-12-31' LIMIT 100",
    billing_project_id="desafio-pic",
)

Downloading: 100%|██████████|


### Análise de colunas 

In [4]:
df_chamados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id_chamado                        100 non-null    object             
 1   id_origem_ocorrencia              100 non-null    object             
 2   data_inicio                       100 non-null    datetime64[us]     
 3   data_fim                          100 non-null    datetime64[us]     
 4   id_bairro                         100 non-null    object             
 5   id_territorialidade               100 non-null    object             
 6   id_logradouro                     100 non-null    object             
 7   numero_logradouro                 100 non-null    Int64              
 8   id_unidade_organizacional         100 non-null    object             
 9   nome_unidade_organizacional       100 non-null    object          

### Análise de dados duplicados
Nenhum registro duplicado foi encontrado

In [5]:
df_chamados.duplicated().sum()

np.int64(0)

### Análise de dados nulos
Foram encontradas as seguintes colunas com dados nulos:
- numero_logradouro: 28
- longitude: 423
- latitude: 28
- data_alvo_diagnostico: 1000
- data_real_diagnostico: 1000
- justificativa_status: 965

In [6]:
print(
    [
        (col, df_chamados[col].isna().sum())
        for col in df_chamados.columns[df_chamados.isna().any()].tolist()
    ]
)

[('longitude', np.int64(63)), ('latitude', np.int64(63)), ('data_alvo_diagnostico', np.int64(100)), ('data_real_diagnostico', np.int64(100)), ('justificativa_status', np.int64(100))]


### Análise descritiva dos dados

In [7]:
df_chamados.describe().T

,count,mean,min,25%,50%,75%,max,std
data_inicio,100,2024-08-15 20:48:40.130000,2024-08-01 09:00:03,2024-08-02 14:28:50.250000,2024-08-14 23:53:23,2024-08-27 11:36:15,2024-08-30 15:50:01,NaN
data_fim,100,2024-08-18 14:54:38.160000,2024-08-01 16:34:04,2024-08-05 14:27:49.750000,2024-08-16 12:16:06,2024-08-30 10:01:28.250000,2024-09-05 21:19:57,NaN
numero_logradouro,100.0,151.11,0.0,38.25,93.5,161.25,1500.0,221.62453
longitude,37.0,-43.180186,-43.205449,-43.182463,-43.179153,-43.179045,-43.108415,0.013414
latitude,37.0,-22.897603,-22.912852,-22.9056,-22.900908,-22.900293,-22.763595,0.026391
data_alvo_finalizacao,100,2024-09-01 01:55:57.600000,2024-08-12 09:00:00,2024-08-13 14:28:00,2024-09-04 23:52:30,2024-09-10 15:05:00,2024-09-17 16:29:00,NaN
data_alvo_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
data_real_diagnostico,0,NaT,NaT,NaT,NaT,NaT,NaT,NaN
tempo_prazo,100.0,11.16,7.0,7.0,15.0,15.0,15.0,4.016934
reclamacoes,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Tratamento de dados nulos
Considerando as análises iniciais, considerei substituir os valores nulos de:
- colunas Int ou Float pela mediana
- colunas de latitude ou longitude pela moda
- colunas de datas não serão alteradas, pois todos os registros são nulos e até o momento não serão usados
- colunas object não serão alteradas, pois até o momento não serão usadas

#### Coluna numero_logradouro

In [8]:
df_chamados["numero_logradouro"] = df_chamados["numero_logradouro"].fillna(
    df_chamados["numero_logradouro"].median()
)
df_chamados["numero_logradouro"].isna().sum()

np.int64(0)

### Coluna longitude

In [9]:
longitude_moda = df_chamados["longitude"].mode()
if not longitude_moda.empty:
    df_chamados["longitude"] = df_chamados["longitude"].fillna(
        value=float(longitude_moda.iloc[0]),
    )
df_chamados["longitude"].isna().sum()

np.int64(0)

#### Coluna latitude

In [10]:
latitude_moda = df_chamados["latitude"].mode()
if not latitude_moda.empty:
    df_chamados["latitude"] = df_chamados["latitude"].fillna(
        value=float(latitude_moda.iloc[0]),
    )
df_chamados["latitude"].isna().sum()

np.int64(0)

### Tratamento de datas
Para fazer as requisições à API do Open-Meteo é preciso ter a data formatada como "YYYY-MM-dd"

#### Coluna data_inicio

In [11]:
df_chamados["data_inicio"] = pd.to_datetime(df_chamados["data_inicio"]).dt.strftime(
    "%Y-%m-%d"
)
df_chamados["data_inicio"].head()

0    2024-08-27
1    2024-08-22
2    2024-08-22
3    2024-08-22
4    2024-08-27
Name: data_inicio, dtype: object

#### Coluna data_fim

In [12]:
df_chamados["data_fim"] = pd.to_datetime(df_chamados["data_fim"]).dt.strftime(
    "%Y-%m-%d"
)
df_chamados["data_fim"].head()

0    2024-08-30
1    2024-08-23
2    2024-09-03
3    2024-08-23
4    2024-08-28
Name: data_fim, dtype: object

## Baixando dados da API Open-Meteo

## Acessando a API

In [18]:
from typing import Any


df = df_chamados.copy()

API_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_FIELDS = "temperature_2m_max,temperature_2m_min,precipitation_sum"
TIMEOUT_SECONDS = 20

# Colunas de saída
for col in [
    "temperatura_maxima",
    "temperatura_minima",
    "precipitacao",
]:
    df[col] = pd.NA


def is_missing_coords(row: pd.Series) -> bool:
    return bool(pd.isna([row["latitude"], row["longitude"]]).any())


def build_weather_params(row: pd.Series) -> dict[str, Any]:
    return {
        "latitude": float(row["latitude"]),
        "longitude": float(row["longitude"]),
        "start_date": row["data_inicio"],
        "end_date": row["data_fim"],
        "daily": DAILY_FIELDS,
        "timezone": "auto",
    }


def fetch_daily_temperatures(
    session: requests.Session,
    params: dict[str, Any],
) -> tuple[list[Any], list[Any], list[Any]]:
    response = session.get(API_URL, params=params, timeout=TIMEOUT_SECONDS)
    response.raise_for_status()

    daily = response.json().get("daily", {})
    max_values = daily.get("temperature_2m_max", [])
    min_values = daily.get("temperature_2m_min", [])
    precipitacao = daily.get("precipitation_sum", [])
    return max_values, min_values, precipitacao


session = requests.Session()
for index, row in df.iterrows():
    if is_missing_coords(row):
        continue

    params = build_weather_params(row)

    try:
        max_values, min_values, precipitacao = fetch_daily_temperatures(session, params)
    except requests.RequestException:
        continue

    df.loc[index, "temperatura_maxima"] = max_values[0] if max_values else pd.NA
    df.loc[index, "temperatura_minima"] = min_values[0] if min_values else pd.NA
    df.loc[index, "precipitacao"] = precipitacao[0] if precipitacao else pd.NA


df[
    [
        "temperatura_maxima",
        "temperatura_minima",
        "precipitacao",
    ]
].head(30)

,temperatura_maxima,temperatura_minima,precipitacao
0,20.1,15.5,0.0
1,30.1,18.6,0.0
2,30.0,18.5,0.0
3,30.1,18.6,0.0
4,20.1,15.5,0.0
5,25.2,18.2,0.0
6,19.3,16.5,0.6
7,24.6,17.1,0.0
8,34.3,21.1,0.0
9,23.5,15.6,0.0
